# 🍎 AlphaApple Training (Colab)

**목표**: 170개 셀 전부 제거 (100%)

**현재**:
- 사람 최고: 130개 (76.5%)
- AI 베스트: 110개 (64.6%)

**전략**:
1. 역방향 생성으로 100% 제거 가능한 보드만 생성
2. 경량 모델 (706K params)
3. Behavior Cloning → PPO Fine-tuning

## 🔧 Setup

In [ ]:
# Colab 환경 확인
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Colab")
except:
    IN_COLAB = False
    print("❌ Not in Colab")

# GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GitHub에서 코드 가져오기
if IN_COLAB:
    !git clone https://github.com/kbsooo/AlphaApple.git
    %cd AlphaApple
    !git checkout claude/understand-project-011CUpMBTTDEN1xuP4kCMRKV
else:
    import os
    os.chdir('/home/user/AlphaApple')

In [ ]:
# 의존성 설치
!pip install -q gymnasium numpy torch tqdm

## 📊 1. 전문가 데이터 생성

**역방향 생성기로 100% 제거 가능한 보드 생성**

In [ ]:
import sys
import numpy as np
import pickle
from tqdm.notebook import tqdm

sys.path.insert(0, '.')

from envs.fruitbox_env import FruitBoxEnv, FruitBoxConfig
from envs.backward_generator import BackwardBoardGenerator
from envs.autoregressive_wrapper import make_autoregressive_env

In [ ]:
def collect_expert_data_100pct(n_episodes=500, target_coverage=0.95):
    """
    역방향 생성으로 높은 coverage 보드 생성
    target_coverage=0.95: 95% 제거 가능 (현실적 목표)
    """
    episodes = []
    total_rewards = []
    
    for i in tqdm(range(n_episodes), desc="Collecting expert data"):
        # 역방향 생성
        generator = BackwardBoardGenerator(rows=10, cols=17, seed=i)
        board, solution = generator.generate(target_coverage=target_coverage)
        
        # 환경에 설정
        wrapped_env = make_autoregressive_env(rows=10, cols=17)
        env = wrapped_env.env
        env.board = board.astype(np.int16)
        obs = board.clip(0, 9).astype(np.int8)
        
        observations = []
        actions = []
        rewards = []
        
        episode_reward = 0
        steps = 0
        
        # 작은 것 우선 전략으로 플레이
        while True:
            observations.append(obs.copy())
            
            legal = env.legal_actions()
            if len(legal) == 0:
                break
            
            # 가장 작은 직사각형 선택
            sizes = [(env.rects[a][2]-env.rects[a][0]+1) * (env.rects[a][3]-env.rects[a][1]+1) for a in legal]
            action_idx = legal[np.argmin(sizes)]
            r1, c1, r2, c2 = env.rects[action_idx]
            
            actions.append((r1, c1, r2, c2))
            
            # Step
            obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
            
            rewards.append(reward)
            episode_reward += reward
            steps += 1
            
            if terminated or truncated or steps >= 500:
                break
        
        episodes.append({
            'observations': np.array(observations),
            'actions': np.array(actions),
            'rewards': np.array(rewards),
            'total_reward': episode_reward,
            'steps': steps,
            'seed': i,
        })
        
        total_rewards.append(episode_reward)
    
    print(f"\n=== 수집 완료 ===")
    print(f"에피소드 수: {n_episodes}")
    print(f"평균 보상: {np.mean(total_rewards):.1f} ± {np.std(total_rewards):.1f}")
    print(f"최대 보상: {max(total_rewards):.0f}")
    print(f"총 transition: {sum(ep['steps'] for ep in episodes)}")
    
    return episodes

In [ ]:
# 데이터 수집 (500 episodes, ~5분)
expert_data = collect_expert_data_100pct(n_episodes=500, target_coverage=0.95)

# 저장
with open('expert_data_95pct.pkl', 'wb') as f:
    pickle.dump(expert_data, f)

print("✅ 데이터 저장 완료")

## 🧠 2. 경량 모델 로드

In [ ]:
from models.lightweight_policy import LightweightPolicy

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 모델 생성
policy = LightweightPolicy(rows=10, cols=17, latent_dim=128)
policy = policy.to(device)

print(f"파라미터 수: {sum(p.numel() for p in policy.parameters()):,}")
print(f"Device: {device}")

## 📚 3. Behavior Cloning

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ExpertDataset(Dataset):
    def __init__(self, episodes):
        self.observations = []
        self.actions = []
        
        for ep in episodes:
            for t in range(len(ep['observations'])):
                self.observations.append(ep['observations'][t])
                self.actions.append(ep['actions'][t])
        
        self.observations = np.array(self.observations)
        self.actions = np.array(self.actions)
    
    def __len__(self):
        return len(self.observations)
    
    def __getitem__(self, idx):
        obs = torch.from_numpy(self.observations[idx]).float().unsqueeze(0)  # (1, 10, 17)
        act = torch.from_numpy(self.actions[idx]).long()  # (4,)
        return obs, act

# 데이터셋 생성
dataset = ExpertDataset(expert_data)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Behavior Cloning 학습
optimizer = optim.Adam(policy.parameters(), lr=3e-4)
n_epochs = 30
best_val_loss = float('inf')

for epoch in range(n_epochs):
    # Train
    policy.train()
    train_loss = 0
    
    for batch_obs, batch_act in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False):
        batch_obs = batch_obs.to(device)
        batch_act = batch_act.to(device)
        
        action_tuple = tuple(batch_act[:, i] for i in range(4))
        _, log_prob, _, _ = policy(batch_obs, action=action_tuple)
        
        loss = -log_prob.mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # Val
    policy.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch_obs, batch_act in val_loader:
            batch_obs = batch_obs.to(device)
            batch_act = batch_act.to(device)
            
            action_tuple = tuple(batch_act[:, i] for i in range(4))
            _, log_prob, _, _ = policy(batch_obs, action=action_tuple)
            
            loss = -log_prob.mean()
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    
    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(policy.state_dict(), 'bc_policy_best.pt')
        print(f"  ✅ Best model saved")

print("\n✅ Behavior Cloning 완료!")

## 🎯 4. 평가

In [ ]:
# Best 모델 로드
policy.load_state_dict(torch.load('bc_policy_best.pt'))
policy.eval()

# 평가
def evaluate(policy, n_episodes=50, use_backward=True, target_coverage=0.95):
    episode_rewards = []
    
    with torch.no_grad():
        for i in tqdm(range(n_episodes), desc="Evaluating"):
            if use_backward:
                generator = BackwardBoardGenerator(rows=10, cols=17, seed=10000+i)
                board, _ = generator.generate(target_coverage=target_coverage)
                wrapped_env = make_autoregressive_env(rows=10, cols=17)
                env = wrapped_env.env
                env.board = board.astype(np.int16)
                obs = board.clip(0, 9).astype(np.int8)
                info = {'action_mask': env._compute_action_mask(env.board)}
            else:
                wrapped_env = make_autoregressive_env(rows=10, cols=17)
                obs, info = wrapped_env.reset(seed=10000+i)
            
            episode_reward = 0
            steps = 0
            
            while True:
                obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).unsqueeze(0).to(device)
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=True)
                
                r1 = int(action_tuple[0][0].item())
                c1 = int(action_tuple[1][0].item())
                r2 = int(action_tuple[2][0].item())
                c2 = int(action_tuple[3][0].item())
                
                obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
                
                episode_reward += reward
                steps += 1
                
                if terminated or truncated or steps >= 500:
                    break
            
            episode_rewards.append(episode_reward)
    
    return episode_rewards

# 평가 (역방향 생성 보드)
print("\n=== 95% 제거 가능 보드 평가 ===")
results_95 = evaluate(policy, n_episodes=50, use_backward=True, target_coverage=0.95)
print(f"평균: {np.mean(results_95):.1f} ± {np.std(results_95):.1f}")
print(f"최대: {max(results_95):.0f}/170 ({max(results_95)/170*100:.1f}%)")
print(f"범위: [{min(results_95):.0f}, {max(results_95):.0f}]")

# 평가 (일반 보드)
print("\n=== 일반 보드 평가 ===")
results_normal = evaluate(policy, n_episodes=50, use_backward=False)
print(f"평균: {np.mean(results_normal):.1f} ± {np.std(results_normal):.1f}")
print(f"최대: {max(results_normal):.0f}/170 ({max(results_normal)/170*100:.1f}%)")

## 💾 5. 모델 다운로드 (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # 모델 다운로드
    files.download('bc_policy_best.pt')
    print("✅ 모델 다운로드 완료")

## 📊 결과 요약

**목표**: 170개 전부 제거

**베이스라인**:
- 사람 최고: 130개 (76.5%)
- Greedy: 110개 (64.6%)

**현재 모델**:
- 일반 보드: ??? 개
- 95% 보드: ??? 개 (목표에 가까워야 함)

**Next Steps**:
1. PPO Fine-tuning (추가 개선)
2. target_coverage 조정 (0.95 → 1.0?)
3. 모델 크기 조정